# Clear dataset schemas, contents, and metadata descriptions

Good for resetting before another test.

In [0]:
from pprint import pprint as pp

import dataiku
from dataikuapi.dss.dataset import DSSDatasetSettings

# Scope: Current Project only
client = dataiku.api_client()
project = client.get_default_project()
datasets = project.list_datasets()
print(f"Found {len(datasets)} total datasets in project {project.project_key}")

In [0]:

    
def clear_dataset_schema(dataset_handle):
    try:        
        dataset_handle.set_schema({"columns": []})
    except Exception as e:
        print(f"An error occurred while clearing the schema {e}")
        
def is_dataset_schema_empty(dataset_handle):
    """x"""
    settings = dataset_handle.get_settings()
    column_schema = settings.get_raw().get('schema',{}).get('columns',[])
    return not column_schema

def is_plugin_dataset(ds_name):
    return ds_name.startswith("dss_") and ds_name.endswith(("_live", "_cached"))

def reset_dataset_settings_to_autodetect(dataset_handle):
    """x"""
    settings = dataset_handle.autodetect_settings()
    # settings is now an object containing the "suggested" new dataset settings, including the detected format
    # and completed schema
    # We can just save the new settings in order to "accept the suggestion"
    settings.save()


In [0]:
def clear_dataset_description(dataset_handle):
    """x"""
    try:
        md = dataset_handle.get_metadata() # returns a dict.
#         print(md)
        md['description'] = 'LONG_RESET2'
        md['shortDesc'] = 'SHORT_RESET4'
#         # apply the changes
        dataset_handle.set_metadata(md)
#         print(f"Successfully cleared description.")
        
#         # Access the dataset's settings
        settings = dataset_handle.get_settings()
        # pp(settings.get_raw())

#         # Clear the dataset's description
        settings.description = "LONG_RESET"
        settings.shortdesc   = "SHORT_RESET1"
        settings.shortDesc   = "SHORT_RESET2"
        settings.shortDescription= "SHORT_RESET3"
        # set-metadata {"shortDesc":"new_SHORT_VALUE","description":"LONG_RESET","customFields":{},"tags":[]}
        
        
#         # Save the settings back to the dataset
        settings.save()
        print(f"[SUCCESS] clear_dataset_description")
    except Exception as e:
        print(f"[EXCEPTION] clear_dataset_description: {e}")
    
for ds_item in datasets:
    ds_name = ds_item['name']
    
    if not is_plugin_dataset(ds_name):
        # print(f"Skipping {ds_name} since is not in scope.")
        continue
    dataset = project.get_dataset(ds_name)
    
    print(f"Attempting to clear description for dataset {ds_name}")
    clear_dataset_description(dataset)
    
#     # 1. Clear the Schema
#     try:
#         if is_dataset_schema_empty(dataset):
#             print(f"Schema seems to already be empty for {ds_name}, skipping.")
#         else:
#             clear_dataset_schema(dataset)
#             print(f"Schema cleared for dataset {ds_name}")
#         print(f"Attempting to reset dataset settings for {ds_name}")
#         reset_dataset_settings_to_autodetect(dataset)
#         print(f"Successfully reset dataset settings for {ds_name}")        
#     except Exception as e:
#         print(f"  EXCEPTION: Failed to clear schema for {ds_name}. Reason: {e}")

#     # 2. Clear the Dataset (Data)
#     try:
#         # The clear() method throws an exception if the dataset type 
#         # does not support clearing (e.g., some read-only external inputs)
#         dataset.clear()
#         print(f"  Data cleared for {ds_name}")
#     except Exception as e:
#         # Capture specific API errors regarding capability
#         print(f"  EXCEPTION: Could not clear data (may not be supported for this type). Reason: {e}")

    print("-" * 30)

In [0]:
# # Tamara wrote this function and it makes something pop up in the Web interface before
# # you can continue. 
# # It also does it for ALL datasets, not just one.
# def detect_and_apply_schema(dataset_name, project_key=None):
#     try:
#         # Initialize client
#         client = dataiku.api_client()
        
#         # Get the project
#         if not project_key:
#             project = client.get_default_project()
#         else:
#             project = client.get_project(project_key)
        
#         # Get the flow object
#         flow = project.get_flow()
        
#         # Start the consistency check tool
#         tool = flow.start_tool(type="CHECK_CONSISTENCY")
        
#         # Configure options to check data consistency only
#         options = {
#             "recheckAll": False,
#             "datasets": {"consistencyWithData": True},
#             "recipes": {"schemaConsistency": False, "otherExpensiveChecks": False},
#         }
        
#         # Run the check
#         future = tool.update(options)
#         print(f"Starting consistency check for {dataset_name}, waiting for it to finish...")
        
#         # Wait for the result
#         result = future.wait_for_result()
        
#         print(f"Schema check completed for dataset: {dataset_name}")
#         print(f"Check results: {result}")
        
#         return result
        
#     except Exception as e:
#         print(f"Error detecting schema for {dataset_name}: {str(e)}")
#         return None

# result = detect_and_apply_schema("dss_meanings_live")
# print(result)

# print("finished all")